# Resources

https://quantum.cloud.ibm.com/docs/en/guides/quick-start
https://quantum.cloud.ibm.com/docs/en/guides/hello-world

# Hardware Access

Only needs to be submitted once

In [ ]:
# from qiskit_ibm_runtime import QiskitRuntimeService

# QiskitRuntimeService.save_account(
# token="DNY-1hx1yw1TAXgfvWO72JeBPmmHVaUnAHyS-8uyiCcu",
# instance="crn:v1:bluemix:public:quantum-computing:us-east:a/2ee71a743276426b8602d449f5e2e034:b5f69e90-11fc-4e77-8458-51024332ae61::",
# )

# Imports

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

# Tools

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(simulator=False, operational=True)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

sim_sampler = StatevectorSampler()
sampler = Sampler(mode=backend)

The pass manager converts the submitted circuit to an Instructio Set Architecture (ISA) circuit that conforms to the backend device's ISA, accounting for the device's basis gates and qubit connectivity.

# Bell State Circuit

In [ ]:
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure_all()

result = sim_sampler.run([qc_bell], shots=1024).result()
print(result[0].data.meas.get_counts())

The sampler calculates probabilities or quasi-probabilities (can be negative) of bitstrings from quantum circuits. Allows for high-level user to write algorithms without worrying about hadware details.

StatevectorSampler converts circuits into pure state vectors. A state vector is the state of a system described by a state space representation and contains all possible information about the system.

In [ ]:
counts = result[0].data.meas.get_counts()
plot_histogram(counts)

### Real Hardware

In [ ]:
isa_qc_bell = pm.run(qc_bell)

job = sampler.run([isa_qc_bell], shots=1024)

print((f"Job ID: {job.job_id()}"))
print(f"Job Status: {job.status()}")

In [ ]:
job_result = job.result()
counts = job_result[0].data.meas.get_counts()
plot_histogram(counts)

# GHZ State Circuit

In [ ]:
qc_ghz = QuantumCircuit(3)
qc_ghz.h(0)
qc_ghz.cx(0, 1)
qc_ghz.cx(1, 2)
qc_ghz.measure_all()

result = sim_sampler.run([qc_ghz], shots=1024).result()
counts = result[0].data.meas.get_counts()
plot_histogram(counts)

### Adding an X-gate

In [ ]:
qc_ghz_x1 = QuantumCircuit(3)
qc_ghz_x1.h(0)
qc_ghz_x1.cx(0, 1)
qc_ghz_x1.cx(1, 2)
qc_ghz_x1.x(1)
qc_ghz_x1.measure_all()

result = sim_sampler.run([qc_ghz_x1], shots=1024).result()
counts = result[0].data.meas.get_counts()
plot_histogram(counts)

### Real Hardware

In [ ]:
isa_qc_ghz = pm.run(qc_ghz)

job = sampler.run([isa_qc_ghz], shots=1024)

print((f"Job ID: {job.job_id()}"))
print(f"Job Status: {job.status()}")

In [ ]:
job.wait_for_final_state()

job_result = job.result()
counts = job_result[0].data.meas.get_counts()
plot_histogram(counts)

# Hello World

In a quantum program, quantum circuits are the native formate in which to represent quantum instructions, and operators represent the observables being measured.

In [ ]:
import matplotlib.pyplot as plt
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import (
    EstimatorOptions,
    EstimatorV2 as Estimator,
)

### Write a simple quantum program

In [ ]:
# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a rawing of the circuit
qc.draw("mpl")

Two ways to return data:
- Sampling output for a set of qubits chosen to measure,
- Expectation value of an observable

In [ ]:
# Set up observables
observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

### Optimise the circuit and operators

In [ ]:
# Convert to a ISA circuit
isa_qc = pm.run(qc)

isa_qc.draw("mpl", idle_wires=False)

### Execute using quantum primatives

Quantum computers produce random results, so a sample of outputs must be collected by running the circuit multiple times. The estimator is then used to estimate the value of the observable.

In [ ]:
# Construct the Estimator instance
estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

# Map observables to ISA circuit layout
mapped_observables = [
    observable.apply_layout(isa_qc.layout) for observable in observables
]

Run takes a list of tuples, conining the selection of circuits, observables, and parameters. Each tuple is referred to as a primitive unified bloc (PUB).

In [ ]:
# One pub, with one circuit to run against different observables
job = estimator.run([(isa_qc, mapped_observables)])

print(f"Job ID: {job.job_id()}")

In [ ]:
# The result of the entire submission
# In this instance, the result of one pub and some metadata
job_result = job.result()

# The result of the single pub
# Contains information on all six observables
pub_result = job_result[0]

### Analyse the result

Estimator data is expectation values and errors on observables, Sampler data is bitstrings representing states and their counts.

In [ ]:
expectation_values = pub_result.data.evs
errors = pub_result.data.stds

# Plot graph
plt.bar(observables_labels, expectation_values)
plt.xlabel("Observable")
plt.ylabel("Expectation Value")
plt.show()

## Scaling to a Large Number of Qubits

### Map the problem

Write a function that returns a `QuantumCircuit` that prepares an _n_-qubit GHZ state. Use that fucntion to repare a 100-qubit QHZ state and collect the observables to be measured.

In [ ]:
def generate_GHZ_state_qc(n: int) -> QuantumCircuit:
    """This function generates a qiskit.QuantumCircuit for an n-qubit GHZ state.
    
    Args:
        n (int): Number of qubits in the GHZ state.
        
    Returns:
        QuantumCircuit: Quantum circuit that generates the n-qubit GHZ state, assuming all qubits start in the 0 state."""
    
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input.")
    
    return qc

In [ ]:
# Create a new circuit with 100 qubits in the GHZ state
n = 100
qc = generate_GHZ_state_qc(n=n)

Map to the operators of interest. We use `ZZ` operators between qubits to examine the behaviour as they get further papart. Increasingly iaccurate EVs between distant qubits reveals the level of noise in the machine.

In [ ]:
# ZZII...II, ZIZI...II, ..., ZIII...IZ
operator_strings = [
    "Z" + ("I" * i) + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]

operators = [SparsePauliOp(operator) for operator in operator_strings]

### Optimise for execution on QHW

In [ ]:
backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_qc = pm.run(qc)
isa_operators = [op.apply_layout(isa_qc.layout) for op in operators]

### Execute on hardware

Error suppression can be enabled usig a technique called dynamical decoupling. The resilience level specifies how much resilience to build agaist errors. Higher levels generate more accurate results, at the expense of longer processing times.

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [ ]:
# Submit the circuit to the Estimator
job = estimator.run([(isa_qc, isa_operators)])

print(f"Job ID: {job.job_id()}")

### Process results

In [ ]:
job.wait_for_final_state()

# Retrieve the job data
result = job.result()[0]
values = result.data.evs # Expectaion value for each Z operator
values = [
    v / values[0] for v in values
] # Normalise the Evs to evaluate how they decay with distance

distance = [d for d in range(1, len(operators) + 1)]

# Plot graph
plt.plot(distance, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle$")
plt.legend()
plt.show()

The plot shows that as the distance between qubits increases, the signal decays because of the presence of noise (should be constant between all qubits).